# LeetCode 1056 — Confusing Number bằng Python

**Link đề:** https://leetcode.com/problems/confusing-number/
**Độ khó:** Easy · **Chủ đề:** Math

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

```python
class Solution:
    def confusingNumber(self, n: int) -> bool:
        ...
```

## Mục tiêu học
1. Hiểu phép **xoay 180°** một chữ số và một số: chữ số nào xoay được, xoay thành gì, vì sao thứ tự chữ số bị **đảo ngược**.
2. Dùng `dict` làm **bảng tra** (lookup table) và `in` để kiểm tra hợp lệ.
3. Giải bằng **chuỗi** (dễ hiểu) rồi bằng **toán** `% 10`, `// 10` (không dùng chuỗi).
4. Trace từng chữ số, chứng minh tính đúng, phân tích Time/Space.
5. Tự làm biến thể: số strobogrammatic (xoay vẫn y nguyên), đếm số confusing trong một khoảng.

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài — số "gây nhầm lẫn" khi xoay ngược

Một **confusing number** là số mà khi **xoay 180°** sẽ thành **một số hợp lệ khác** với số ban đầu.

Xoay từng chữ số 180°:
- `0 → 0`, `1 → 1`, `6 → 9`, `8 → 8`, `9 → 6` — **hợp lệ**.
- `2, 3, 4, 5, 7` xoay thành hình không phải chữ số — **không hợp lệ**.

Khi xoay cả số, ta xoay từng chữ số **và** thứ tự chữ số bị đảo (chữ số cuối lên đầu). Có thể bỏ qua các số 0 ở đầu sau khi xoay (ví dụ `8000` xoay thành `0008` coi là `8`).

Cho số nguyên `n`, trả về `True` nếu `n` là confusing number, ngược lại `False`.

| `n` | Xoay 180° | Output | Giải thích |
|---:|---:|:---:|---|
| `6` | `9` | `True` | hợp lệ và khác 6 |
| `89` | `68` | `True` | 9→6 lên đầu, 8→8 |
| `11` | `11` | `False` | xoay vẫn là 11 |
| `25` | — | `False` | `2`, `5` không xoay được |

**Constraints (theo hiểu biết):** `0 <= n <= 10^9`.

Hai điều kiện để `True`:
1. **Mọi** chữ số thuộc `{0, 1, 6, 8, 9}`.
2. Số sau khi xoay **khác** `n`.

In [1]:
# Hình dung: viết 89 lên giấy rồi xoay 180 độ
n = "89"
rot = {"0": "0", "1": "1", "6": "9", "8": "8", "9": "6"}
print("đọc từ phải sang trái:", list(reversed(n)))
print("xoay từng chữ số      :", [rot[d] for d in reversed(n)])
print("số mới                :", "".join(rot[d] for d in reversed(n)))

đọc từ phải sang trái: ['9', '8']
xoay từng chữ số      : ['6', '8']
số mới                : 68


## 2. Nền tảng Python cần biết

### 2.1. `dict` làm bảng tra

`dict` lưu cặp **key → value**. Ở đây key là chữ số gốc, value là chữ số sau khi xoay.

- `rot[k]` đọc value của key `k` (lỗi `KeyError` nếu không có key).
- `k in rot` kiểm tra key có tồn tại không — trung bình `O(1)`.

In [2]:
rot = {0: 0, 1: 1, 6: 9, 8: 8, 9: 6}
print(rot[6])          # 9
print(6 in rot)        # True
print(2 in rot)        # False -> 2 không xoay được
try:
    rot[2]
except KeyError as e:
    print("KeyError với key", e)

9
True
False
KeyError với key 2


### 2.2. Vì sao xoay 180° làm **đảo thứ tự** chữ số?

Xoay 180° = lật ngang + lật dọc. Chữ số **bên phải nhất** sẽ chạy sang **bên trái nhất**. Vì vậy: đọc `n` từ phải sang trái, xoay từng chữ số, ghép lại.

### 2.3. `reversed`, slicing `[::-1]` và `str(n)`

- `str(89)` → `"89"`.
- `s[::-1]` đảo chuỗi: `"89"[::-1]` → `"98"`.
- `reversed(s)` trả một iterator đi ngược; muốn xem thì bọc `list(...)`.
- `int("0008")` → `8`: `int` tự bỏ các số 0 ở đầu.

In [3]:
s = str(8000)
print(s, s[::-1], list(reversed(s)))
print(int("0008"))           # 8
print(int("01"))             # 1

8000 0008 ['0', '0', '0', '8']
8
1


### 2.4. Hàm `all(...)` — kiểm tra "mọi phần tử đều thỏa"

`all(iterable)` trả `True` nếu **mọi** phần tử đúng (và `True` với iterable rỗng). Dừng sớm ngay khi gặp phần tử sai.

In [4]:
valid = set("01689")
print(all(d in valid for d in "8619"))   # True
print(all(d in valid for d in "8629"))   # False vì có '2'

True
False


### 2.5. Lấy chữ số bằng toán: `% 10` và `// 10`

- `n % 10` → chữ số **cuối**.
- `n // 10` → **bỏ** chữ số cuối (chia lấy nguyên).
- Ghép số: `res = res * 10 + d` thêm chữ số `d` vào **bên phải** của `res`.

Đọc chữ số từ cuối và đẩy vào bên phải `res` → tự nhiên đã **đảo thứ tự**. Đúng điều phép xoay cần!

In [5]:
x = 89
print(x % 10, x // 10)      # 9 8
res = 0
for d in [9, 8]:            # chữ số lấy từ cuối lên
    res = res * 10 + d
    print("res =", res)

9 8
res = 9
res = 98


## 3. Cách 1 — Chuỗi + bảng tra (dễ hiểu nhất)

1. `s = str(n)`.
2. Nếu có chữ số không nằm trong bảng → `False`.
3. Tạo chuỗi xoay: duyệt `s` từ phải sang trái, thay từng chữ số.
4. So sánh `int(chuỗi xoay) != n` (dùng `int` để bỏ số 0 ở đầu).

In [6]:
ROT_STR = {"0": "0", "1": "1", "6": "9", "8": "8", "9": "6"}

def confusing_string(n: int) -> bool:
    s = str(n)
    for d in s:
        if d not in ROT_STR:
            return False                     # có chữ số không xoay được
    rotated = "".join(ROT_STR[d] for d in reversed(s))
    return int(rotated) != n

for n in [6, 89, 11, 25, 10, 916, 0, 8000]:
    print(n, "->", confusing_string(n))

6 -> True
89 -> True
11 -> False
25 -> False
10 -> True
916 -> False
0 -> False
8000 -> True


### Nhìn kỹ vài ca đặc biệt

- `10` xoay thành `"01"` = `1` ≠ `10` → `True`.
- `916`: đọc ngược `6,1,9` → xoay `9,1,6` → `916` = chính nó → `False`.
- `0` xoay thành `0` → `False`.
- `8000` xoay thành `"0008"` = `8` ≠ `8000` → `True`.

Nếu so sánh **chuỗi** `rotated != s` thay vì số, `10` cho `"01" != "10"` → `True` cũng đúng; nhưng so số rõ ràng hơn với tinh thần "bỏ số 0 ở đầu".

## 4. Nút thắt & insight: làm không cần chuỗi được không?

Cách chuỗi đúng và đủ nhanh (`n` có tối đa 10 chữ số). Nhưng trong phỏng vấn thường hỏi tiếp: *"không đổi sang chuỗi thì sao?"*

Insight: phép xoay = **đảo số** (giống bài Reverse Integer / Palindrome Number) **nhưng** mỗi chữ số được thay bằng `rot[d]` trước khi ghép:

```text
rotated = rotated * 10 + rot[n % 10]
n = n // 10
```

Thêm một kiểm tra: nếu `n % 10` không nằm trong bảng → trả `False` ngay.

Ta phải **lưu `original = n`** vì `n` sẽ bị chia dần về 0.

In [7]:
# Từng giá trị trung gian khi xoay 89
ROT = {0: 0, 1: 1, 6: 9, 8: 8, 9: 6}
x, rotated = 89, 0
d = x % 10; rotated = rotated * 10 + ROT[d]; x //= 10
print("lấy", d, "-> xoay", ROT[d], "| rotated =", rotated, "| x còn", x)
d = x % 10; rotated = rotated * 10 + ROT[d]; x //= 10
print("lấy", d, "-> xoay", ROT[d], "| rotated =", rotated, "| x còn", x)

lấy 9 -> xoay 6 | rotated = 6 | x còn 8
lấy 8 -> xoay 8 | rotated = 68 | x còn 0


## 5. Cách 2 (lời giải chính) — Toán học, ghép từng đoạn

**Đoạn 1:** `original = n`, `rotated = 0`.

**Đoạn 2:** `while n > 0:` lấy `d = n % 10`; nếu `d not in ROT` → `return False`.

**Đoạn 3:** `rotated = rotated * 10 + ROT[d]`, rồi `n //= 10`.

**Đoạn 4:** `return rotated != original`.

Ca `n = 0`: vòng `while` không chạy, `rotated = 0 = original` → `False`. Đúng (0 xoay vẫn là 0).

In [8]:
ROT = {0: 0, 1: 1, 6: 9, 8: 8, 9: 6}

def confusing_math(n: int) -> bool:
    original = n
    rotated = 0
    while n > 0:
        d = n % 10
        if d not in ROT:
            return False
        rotated = rotated * 10 + ROT[d]
        n //= 10
    return rotated != original

for n in [6, 89, 11, 25, 10, 916, 0, 8000]:
    print(n, "->", confusing_math(n))

6 -> True
89 -> True
11 -> False
25 -> False
10 -> True
916 -> False
0 -> False
8000 -> True


## 6. Trace từng bước

### Ví dụ `n = 89`

| Vòng | `n` trước | `d = n % 10` | Hợp lệ? | `ROT[d]` | `rotated` sau | `n` sau |
|---:|---:|:---:|:---:|:---:|---:|---:|
| 1 | 89 | 9 | có | 6 | 6 | 8 |
| 2 | 8 | 8 | có | 8 | 68 | 0 |

Kết thúc: `68 != 89` → `True`.

### Ví dụ `n = 916`

| Vòng | `n` trước | `d` | `ROT[d]` | `rotated` sau |
|---:|---:|:---:|:---:|---:|
| 1 | 916 | 6 | 9 | 9 |
| 2 | 91 | 1 | 1 | 91 |
| 3 | 9 | 9 | 6 | 916 |

`916 == 916` → `False` (số này xoay vẫn y nguyên — gọi là **strobogrammatic**).

In [9]:
def trace_confusing(n: int) -> bool:
    print(f"== n = {n}")
    original, rotated, step = n, 0, 1
    while n > 0:
        d = n % 10
        if d not in ROT:
            print(f"vòng {step}: d={d} không xoay được -> False")
            return False
        rotated = rotated * 10 + ROT[d]
        n //= 10
        print(f"vòng {step}: d={d} -> {ROT[d]}, rotated={rotated}, n còn={n}")
        step += 1
    ans = rotated != original
    print(f"so sánh {rotated} != {original} -> {ans}")
    return ans

trace_confusing(89)
trace_confusing(916)
trace_confusing(1520)

== n = 89
vòng 1: d=9 -> 6, rotated=6, n còn=8
vòng 2: d=8 -> 8, rotated=68, n còn=0
so sánh 68 != 89 -> True
== n = 916
vòng 1: d=6 -> 9, rotated=9, n còn=91
vòng 2: d=1 -> 1, rotated=91, n còn=9
vòng 3: d=9 -> 6, rotated=916, n còn=0
so sánh 916 != 916 -> False
== n = 1520
vòng 1: d=0 -> 0, rotated=0, n còn=152
vòng 2: d=2 không xoay được -> False


False

Chú ý trace `1520`: chữ số cuối `0` hợp lệ, nhưng chữ số tiếp theo `2` thì không → dừng sớm, trả `False`.

## 7. Cách 3 — `str.translate`: xoay bằng một dòng

Python có `str.maketrans(from, to)` tạo bảng dịch ký tự, và `s.translate(table)` thay mọi ký tự theo bảng. Kết hợp `[::-1]` để đảo.

Cần kiểm tra trước rằng `s` chỉ gồm ký tự hợp lệ, vì `translate` **giữ nguyên** ký tự không có trong bảng (ví dụ `2` vẫn là `2`).

In [10]:
table = str.maketrans("01689", "01986")
print("6189".translate(table))           # 9186
print("25".translate(table))             # 25 (không đổi!) -> phải tự kiểm tra hợp lệ

VALID = set("01689")

def confusing_translate(n: int) -> bool:
    s = str(n)
    if not set(s) <= VALID:              # tập chữ số phải là tập con của VALID
        return False
    return int(s[::-1].translate(table)) != n

for n in [6, 89, 11, 25, 10, 916]:
    print(n, "->", confusing_translate(n))

9186
25
6 -> True
89 -> True
11 -> False
25 -> False
10 -> True
916 -> False


### `set(s) <= VALID` nghĩa là gì?

Với hai `set`, toán tử `<=` kiểm tra **tập con**: mọi phần tử bên trái đều có trong bên phải. `set("8619") <= set("01689")` → `True`.

In [11]:
print(set("8619"), set("8619") <= set("01689"))
print(set("8629") <= set("01689"))

{'1', '9', '8', '6'} True
False


## 8. Code nộp LeetCode

### Bản toán học (lời giải chính)

In [12]:
class Solution:
    def confusingNumber(self, n: int) -> bool:
        rot = {0: 0, 1: 1, 6: 9, 8: 8, 9: 6}
        original, rotated = n, 0
        while n > 0:
            d = n % 10
            if d not in rot:
                return False
            rotated = rotated * 10 + rot[d]
            n //= 10
        return rotated != original

sol = Solution()
assert sol.confusingNumber(6) is True
assert sol.confusingNumber(89) is True
assert sol.confusingNumber(11) is False
assert sol.confusingNumber(25) is False
print("Solution (math) PASS")

Solution (math) PASS


### Bản chuỗi

In [13]:
class SolutionString:
    def confusingNumber(self, n: int) -> bool:
        rot = {"0": "0", "1": "1", "6": "9", "8": "8", "9": "6"}
        s = str(n)
        if any(d not in rot for d in s):
            return False
        return int("".join(rot[d] for d in reversed(s))) != n

assert SolutionString().confusingNumber(89) is True
assert SolutionString().confusingNumber(11) is False
assert SolutionString().confusingNumber(10) is True
print("SolutionString PASS")

SolutionString PASS


`any(...)` là "anh em" của `all(...)`: trả `True` nếu **có ít nhất một** phần tử đúng. `any(d not in rot for d in s)` = "có chữ số nào không xoay được không?".

## 9. Vì sao đúng?

**Invariant của vòng lặp (cách toán):** sau `k` vòng, `n` là số gồm các chữ số còn lại (phần đầu của số gốc), và `rotated` là **kết quả xoay của `k` chữ số cuối** đã bóc ra.

- Mỗi vòng bóc chữ số cuối `d` của phần còn lại — đó là chữ số đứng **ngay bên trái** đoạn đã xoay. Khi xoay 180°, nó phải nằm **ngay bên phải** đoạn đã xoay → `rotated * 10 + rot[d]` đặt đúng chỗ.
- Nếu gặp chữ số không xoay được, cả số không hợp lệ → trả `False` là đúng (điều kiện 1 hỏng).
- Hết vòng: `rotated` là toàn bộ số sau xoay; số 0 ở đầu tự biến mất vì phép `* 10 + 0` lúc đầu cho 0. So với `original` kiểm tra điều kiện 2.

## 10. Độ phức tạp

Gọi `d` = số chữ số của `n` (`d = ⌊log10 n⌋ + 1`, tối đa 10).

| Cách | Time | Space phụ |
|---|---|---|
| 1. Chuỗi + dict | `O(d)` | `O(d)` (chuỗi `s`, chuỗi xoay) |
| 2. Toán `% 10`, `// 10` | `O(d)` | `O(1)` |
| 3. `translate` | `O(d)` | `O(d)` |

Vì `d = O(log n)`, có thể viết Time `O(log n)`.

## 11. Bộ test

Nhóm test:
1. Ví dụ đề: `6`, `89`, `11`, `25`.
2. Biên: `0`, `1`, `10^9` (`1000000000` → xoay thành `1` → `True`).
3. Strobogrammatic (xoay giữ nguyên): `69`, `88`, `916`, `6009` → `False`.
4. Có số 0 cuối: `10`, `8000`, `100` → `True`.
5. Một chữ số không hợp lệ ở giữa: `1201`.
6. Đối chiếu 3 cách trên `0..20000`.

In [14]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:28} PASS -> {actual}")

tests = [
    ("đề 6", 6, True), ("đề 89", 89, True), ("đề 11", 11, False), ("đề 25", 25, False),
    ("0", 0, False), ("1", 1, False), ("10^9", 10**9, True),
    ("strobo 69", 69, False), ("strobo 88", 88, False),
    ("strobo 916", 916, False), ("strobo 6009", 6009, False),
    ("số 0 cuối 10", 10, True), ("8000", 8000, True), ("100", 100, True),
    ("chữ số xấu ở giữa 1201", 1201, False), ("9", 9, True), ("8", 8, False),
]
for name, n, exp in tests:
    check(name, confusing_math(n), exp)

đề 6                         PASS -> True
đề 89                        PASS -> True
đề 11                        PASS -> False
đề 25                        PASS -> False
0                            PASS -> False
1                            PASS -> False
10^9                         PASS -> True
strobo 69                    PASS -> False
strobo 88                    PASS -> False
strobo 916                   PASS -> False
strobo 6009                  PASS -> False
số 0 cuối 10                 PASS -> True
8000                         PASS -> True
100                          PASS -> True
chữ số xấu ở giữa 1201       PASS -> False
9                            PASS -> True
8                            PASS -> False


In [15]:
for n in range(0, 20001):
    a = confusing_string(n)
    b2 = confusing_math(n)
    c = confusing_translate(n)
    assert a == b2 == c, (n, a, b2, c)
print("0..20000: 3 cách khớp nhau")
print("Số confusing trong 1..100:", [n for n in range(1, 101) if confusing_math(n)])

0..20000: 3 cách khớp nhau
Số confusing trong 1..100: [6, 9, 10, 16, 18, 19, 60, 61, 66, 68, 80, 81, 86, 89, 90, 91, 98, 99, 100]


## 12. Lỗi người mới hay gặp

### Lỗi 1: Quên đảo thứ tự — chỉ xoay từng chữ số tại chỗ

```python
rotated = "".join(rot[d] for d in s)    # thiếu reversed
```

Với `89` ra `"86"` ≠ 89 → vẫn `True`, tình cờ đúng kết luận! (Cell dưới cho thấy `6996` cũng tình cờ đúng, còn `69`, `96` thì sai.) Nhưng với `16` ra `"19"` trong khi xoay thật là `"91"`. Với `69`: sai cho `"96"` ≠ 69 → `True`, nhưng thật ra `69` xoay vẫn là `69` → `False`.

In [16]:
def wrong_no_reverse(n):
    rot = {"0": "0", "1": "1", "6": "9", "8": "8", "9": "6"}
    s = str(n)
    if any(d not in rot for d in s):
        return False
    return int("".join(rot[d] for d in s)) != n

for n in [69, 96, 6996]:
    print(n, "sai:", wrong_no_reverse(n), "| đúng:", confusing_math(n))

69 sai: True | đúng: False
96 sai: True | đúng: False
6996 sai: True | đúng: True


### Lỗi 2: Quên lưu `original`

```python
while n > 0: ...
return rotated != n      # n đã thành 0!
```

Sau vòng lặp `n == 0` nên gần như luôn trả `True` (trừ khi `rotated == 0`).

### Lỗi 3: Chỉ kiểm tra "mọi chữ số hợp lệ" mà quên điều kiện "khác số gốc"

Khi đó `11`, `88`, `69` bị trả `True` sai.

### Lỗi 4: Coi `0` là confusing / dùng `while n >= 0` (vòng lặp vô hạn)

`0 // 10 == 0` nên `while n >= 0` không bao giờ dừng. Dùng `while n > 0`.

In [17]:
def wrong_forget_original(n):
    rotated = 0
    while n > 0:
        d = n % 10
        if d not in ROT:
            return False
        rotated = rotated * 10 + ROT[d]
        n //= 10
    return rotated != n          # so với 0

print("11 sai:", wrong_forget_original(11), "| đúng:", confusing_math(11))

11 sai: True | đúng: False


## 13. Mẹo nhận dạng pattern

- "Xoay 180°", "đọc ngược", "soi gương" một số → **đảo chữ số** + **bảng ánh xạ** từng chữ số.
- Bóc chữ số bằng `% 10` / `// 10`, ghép bằng `res * 10 + d` — khuôn chung với Reverse Integer (7), Palindrome Number (9), Add Digits (258).
- Tập ký tự hợp lệ nhỏ, cố định → `dict` / `set` hằng số.

| Bài liên quan | Điểm khác |
|---|---|
| 246. Strobogrammatic Number | xoay xong phải **bằng** số gốc (ngược bài này) |
| 247/248. Strobogrammatic Number II/III | sinh/đếm số strobogrammatic |
| 1088. Confusing Number II | đếm số confusing trong `[1, N]` bằng DFS |
| 788. Rotated Digits | xoay từng chữ số **không đảo** thứ tự |

## 14. Biến thể

### Biến thể A — Strobogrammatic: xoay vẫn y nguyên

Chỉ đổi điều kiện cuối: `rotated == original`. Hoặc dùng **hai con trỏ** trên chuỗi: `s[i]` xoay phải bằng `s[j]` đối xứng.

In [18]:
def is_strobogrammatic(num: str) -> bool:
    rot = {"0": "0", "1": "1", "6": "9", "8": "8", "9": "6"}
    i, j = 0, len(num) - 1
    while i <= j:
        if num[i] not in rot or rot[num[i]] != num[j]:
            return False
        i += 1
        j -= 1
    return True

print([x for x in ["69", "88", "962", "1", "818", "619"] if is_strobogrammatic(x)])

['69', '88', '1', '818', '619']


### Biến thể B — Đếm số confusing trong `[1, N]` (ý của 1088)

Cách ngây thơ: duyệt từng số. Cách tốt hơn: chỉ **sinh** các số dùng chữ số `0,1,6,8,9` bằng DFS (mỗi tầng thêm một chữ số), số lượng ít hơn rất nhiều.

In [19]:
def count_confusing_dfs(N: int) -> int:
    digits = [0, 1, 6, 8, 9]
    count = 0
    def dfs(x):
        nonlocal count
        if x > N:
            return
        if x > 0 and confusing_math(x):
            count += 1
        for d in digits:
            nxt = x * 10 + d
            if nxt == 0:          # tránh lặp vô hạn ở 0
                continue
            if nxt <= N:
                dfs(nxt)
    dfs(0)
    return count

N = 3000
print(count_confusing_dfs(N), sum(confusing_math(x) for x in range(1, N + 1)))

226 226


`nonlocal count` cho phép hàm lồng `dfs` **sửa** biến `count` của hàm bao ngoài (nếu không có, gán `count += 1` sẽ báo lỗi `UnboundLocalError`).

## 15. Bài tập tự luyện (tăng dần)

**Bài 1 (tính tay):** Xoay 180° các số `1689`, `6890`, `8118`. Số nào là confusing?

**Bài 2 (điền chỗ trống):** hoàn thiện `ex_rotate(n)` trả về số sau khi xoay, hoặc `-1` nếu không xoay được.

**Bài 3 (viết lại):** `ex_confusing(n)` dùng `ex_rotate`.

**Bài 4 (biến thể — Rotated Digits, LeetCode 788):** `ex_good_count(n)` — đếm số `x` trong `[1, n]` mà **xoay từng chữ số tại chỗ** (không đảo thứ tự, bảng `0→0,1→1,8→8,2→5,5→2,6→9,9→6`) cho số **hợp lệ và khác** `x`.

**Bài 5 (mini challenge):** `ex_list_confusing(lo, hi)` — liệt kê các số confusing trong `[lo, hi]` theo thứ tự tăng dần.

In [20]:
def ex_rotate(n: int) -> int:
    # TODO 1: original không cần ở đây; tạo rotated = 0
    # TODO 2: while n > 0: lấy d, nếu không hợp lệ return -1
    # TODO 3: rotated = rotated * 10 + rot[d]; n //= 10
    pass

# assert ex_rotate(89) == 68
# assert ex_rotate(25) == -1
# assert ex_rotate(8000) == 8

In [21]:
def ex_confusing(n: int) -> bool:
    # TODO: r = ex_rotate(n); trả True khi r hợp lệ và khác n
    pass

# assert ex_confusing(6) and ex_confusing(89) and not ex_confusing(11) and not ex_confusing(25)

In [22]:
def ex_good_count(n: int) -> int:
    # TODO: với mỗi x trong 1..n: có chữ số 3,4,7 -> loại; cần ít nhất một chữ số thuộc 2,5,6,9
    pass

# assert ex_good_count(10) == 4      # 2, 5, 6, 9

In [23]:
def ex_list_confusing(lo: int, hi: int) -> list[int]:
    # TODO: list comprehension với điều kiện confusing
    pass

# assert ex_list_confusing(1, 20) == [6, 9, 10, 16, 18, 19]

## 16. Gợi ý 3 mức

**Bài 1:** (1) Đọc ngược rồi xoay từng chữ số. (2) `1689` → ngược `9861` → xoay `6891`. (3) `6890` → `0986` → xoay `0689` = `689`; `8118` → `8118` (giữ nguyên).

**Bài 2:** (1) Như cách 2 nhưng trả `rotated`. (2) Trả `-1` khi `d not in rot`. (3) Ca `n = 0` trả `0`.

**Bài 3:** (1) Hai điều kiện. (2) `r != -1`. (3) `return r != -1 and r != n`.

**Bài 4:** (1) Không đảo thứ tự nên xoay khác số gốc ⇔ có ít nhất một chữ số "đổi hình" (2,5,6,9). (2) Duyệt `str(x)`. (3) `if any(c in "347" for c in s): continue` rồi `if any(c in "2569" for c in s): count += 1`.

**Bài 5:** (1) Dùng lại hàm. (2) `[x for x in range(lo, hi + 1) if ...]`. (3) Điều kiện `confusing(x)`.

## 17. Đáp án tham khảo

Tự làm trước rồi mới xem. Bài 1: `1689 → 6891` (confusing), `6890 → 689` (confusing), `8118 → 8118` (không).

In [24]:
def ans_rotate(n: int) -> int:
    rot = {0: 0, 1: 1, 6: 9, 8: 8, 9: 6}
    rotated = 0
    while n > 0:
        d = n % 10
        if d not in rot:
            return -1
        rotated = rotated * 10 + rot[d]
        n //= 10
    return rotated

def ans_confusing(n: int) -> bool:
    r = ans_rotate(n)
    return r != -1 and r != n

def ans_good_count(n: int) -> int:
    count = 0
    for x in range(1, n + 1):
        s = str(x)
        if any(c in "347" for c in s):
            continue
        if any(c in "2569" for c in s):
            count += 1
    return count

def ans_list_confusing(lo: int, hi: int) -> list[int]:
    return [x for x in range(lo, hi + 1) if ans_confusing(x)]

assert ans_rotate(89) == 68 and ans_rotate(25) == -1 and ans_rotate(8000) == 8
assert ans_rotate(1689) == 6891 and ans_rotate(6890) == 689 and ans_rotate(8118) == 8118
assert ans_confusing(6) and ans_confusing(89) and not ans_confusing(11) and not ans_confusing(25)
assert ans_good_count(10) == 4
assert ans_list_confusing(1, 20) == [6, 9, 10, 16, 18, 19]
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def confusingNumber(self, n: int) -> bool:
        rot = {0: 0, 1: 1, 6: 9, 8: 8, 9: 6}
        original, rotated = n, 0
        while n > 0:
            d = n % 10                     # bóc chữ số cuối
            if d not in rot:
                return False               # không xoay được
            rotated = rotated * 10 + rot[d]  # đặt vào bên phải -> tự đảo thứ tự
            n //= 10
        return rotated != original
```

**Checklist:**
- [ ] Bảng xoay đúng: `6↔9`, `0,1,8` giữ nguyên; `2,3,4,5,7` không hợp lệ.
- [ ] Có **đảo thứ tự** chữ số (xoay 180° chứ không phải xoay từng chữ tại chỗ).
- [ ] Lưu `original` trước khi chia `n`.
- [ ] Hai điều kiện: hợp lệ **và** khác số gốc.
- [ ] Test `0 → False`, `10 → True`, `69 → False`, `916 → False`.
- [ ] Time `O(log n)`, Space `O(1)`.